In [1]:
import sys
import datetime as dt
sys.path.insert(0,'../..')
import matplotlib.pyplot as plt
from rivapy.instruments import ZeroBondSpecification, FixedRateBondSpecification, FloatingRateBondSpecification
from rivapy.instruments.factory import create
from rivapy.tools.enums import DayCounterType, RollConvention, SecuritizationLevel, Currency
from rivapy.pricing import bond_pricing
from rivapy.tools import Schedule, Period
from IPython.display import display, HTML
from rivapy.marketdata import DiscountCurveParametrized, ConstantRate
from rivapy.pricing import DeterministicCashflowPricer
display(HTML("<style>.container { width:80% !important; }</style>"))

%load_ext autoreload
%autoreload 2

%matplotlib inline

c:\Users\GunnarSalentin\OneDrive - RIVACON GmbH\Dokumente\git\RiVaPy\notebooks_draft\instruments\../..\rivapy\__init__.py:13: UserWarning: The pyvacon module is not available. You may not use all functionality without this module. Consider installing pyvacon.
  warnings.warn("The pyvacon module is not available. You may not use all functionality without this module. Consider installing pyvacon.")


# Bonds

Bonds are financial instruments that are issued by governments, banks, and corporates to finance their acitivities. Bonds are fixed income instruments that provide regular interest payments to the bondholders.

**Plain vanilla bonds** are the most simple form of bonds. At the issue date investors pay the notional amount $N$ to the issuer which is paid back upon maturity. During the life of the bond, the investors receives regularly interest payments. The number of interest rate payments depends on the bond's payment frequency. The amount of each interest rate payment is derived from a fixed annual rate $r$, the bond's notional and the length of each interest period given as a year fraction governed by the bond's day count convention. 

In [2]:
# Setting up a plain vanilla (fixed rate) bond
# some key variables first
notional = 100000
maturity_date = dt.datetime(2030, 6, 15)
issue_date = dt.datetime(2020, 6, 15)
coupon_rate = 0.05
period ="1Y"
currency = "EUR"
issuer = "DummyIssuer"
PVBond = FixedRateBondSpecification(obj_id="PVBond",
                                     notional=notional,
                                     issue_date=issue_date,
                                     maturity_date=maturity_date,
                                     coupon=coupon_rate,
                                     issuer=issuer,
                                     currency=currency,
                                     frequency=period,
                                     business_day_convention=RollConvention.UNADJUSTED)
schedule = PVBond.get_schedule()
print(schedule.generate_dates(True))

[datetime.datetime(2021, 6, 15, 0, 0), datetime.datetime(2022, 6, 15, 0, 0), datetime.datetime(2023, 6, 15, 0, 0), datetime.datetime(2024, 6, 15, 0, 0), datetime.datetime(2025, 6, 15, 0, 0), datetime.datetime(2026, 6, 15, 0, 0), datetime.datetime(2027, 6, 15, 0, 0), datetime.datetime(2028, 6, 15, 0, 0), datetime.datetime(2029, 6, 15, 0, 0), datetime.datetime(2030, 6, 15, 0, 0)]


After payment of the notional amount, the value of the bond to the investor is the discounted value of the interest rate cashflows plus the final notional amount: 

$$ PV = \sum_i^n N*r*\delta_i * DF_i + N*DF_n = N * (\sum_i^n r*\delta_i * DF_i + DF_n) $$

with
* $n$: Number or interest rate periods, with the end of period n being the maturity of the instrument
* $N$: Notional amount
* $c$: Annual coupon
* $\delta_i$: year fraction of interest period $i$
* $DF_i$: Discount factor as of $t_i$, i.e. the end date of period $i$.

In [3]:
# Pricing the bond
pricing_date = dt.datetime(2029, 6, 15)
dc = DiscountCurveParametrized("", pricing_date, ConstantRate(0.05), comp_freq=PVBond.nr_annual_payments)
pricer = DeterministicCashflowPricer(pricing_date, PVBond, dc)
print(pricer.expected_cashflows())

# calculate present value of cashflows
bond_price = pricer.pv_cashflows()
print(f"Bond price on {pricing_date} is {bond_price:.2f} {currency}")

print(f"DF: {dc.value(pricing_date,  maturity_date)} ")
print(f"Rate: {dc.value_rate(pricing_date,  maturity_date)} ")
print(dc.value(pricing_date, maturity_date)*105000)
print(1/dc.value(pricing_date, maturity_date))

pricing_date2 = dt.datetime(2025, 6, 15)
dc2 = DiscountCurveParametrized("", pricing_date2, ConstantRate(0.05))
pricer2 = DeterministicCashflowPricer(pricing_date2, PVBond, dc2)
print(pricer2.expected_cashflows())

# calculate present value of cashflows
bond_price = pricer2.pv_cashflows()
print(f"Bond price on {pricing_date} is {bond_price:.2f} {currency}")

End dates of notional structure are not set.
End dates of notional structure are not set.
End dates of notional structure are not set.
End dates of notional structure are not set.


[(datetime.datetime(2020, 6, 15, 0, 0), -100000), (datetime.datetime(2021, 6, 15, 0, 0), 5000.0), (datetime.datetime(2022, 6, 15, 0, 0), 5000.0), (datetime.datetime(2023, 6, 15, 0, 0), 5000.0), (datetime.datetime(2024, 6, 15, 0, 0), 5000.0), (datetime.datetime(2025, 6, 15, 0, 0), 5000.0), (datetime.datetime(2026, 6, 15, 0, 0), 5000.0), (datetime.datetime(2027, 6, 15, 0, 0), 5000.0), (datetime.datetime(2028, 6, 15, 0, 0), 5000.0), (datetime.datetime(2029, 6, 15, 0, 0), 5000.0), (datetime.datetime(2030, 6, 15, 0, 0), 5000.0), (datetime.datetime(2030, 6, 15, 0, 0), 100000.0)]
Bond price on 2029-06-15 00:00:00 is 100000.00 EUR
DF: 0.9523809523809523 
Rate: 0.05 
100000.0
1.05
[(datetime.datetime(2020, 6, 15, 0, 0), -100000), (datetime.datetime(2021, 6, 15, 0, 0), 5000.0), (datetime.datetime(2022, 6, 15, 0, 0), 5000.0), (datetime.datetime(2023, 6, 15, 0, 0), 5000.0), (datetime.datetime(2024, 6, 15, 0, 0), 5000.0), (datetime.datetime(2025, 6, 15, 0, 0), 5000.0), (datetime.datetime(2026, 6, 1

In the given example the bond is priced as of the issue date which results in a value equal to the face value of the bond. During the life of the bond valuations 




* About trading
* about conventions
* about types
* * no intermediate payments: zerobond
* * plainvanillacouponbond
* * floating bond
* * dirty vs clean prices 
* * z-spread / yield / ...
* * features
* * * amortizing
* * * individual coupons
* potential default risk
* * spreads and margin
* risk
* * rho
* * 

In [ ]:
from typing import Callable
import datetime as dt
import pandas as pd
import rivapy.tools.interfaces as interfaces
from rivapy.instruments.factory import create as _instrument_create
from rivapy.marketdata.factory import create as _marketdata_create

class MemoryStorage:
    def __init__(self, create: Callable[[dict], object]):
        self._store = {}
        self.create = create
        
    def add(self, instrument: interfaces.FactoryObject):
        if instrument.obj_id in self._store.keys():
            tmp = self._store[instrument.obj_id][-1]
            ins_dict = instrument.to_dict()
            if interfaces.FactoryObject.hash_for_dict(tmp) != interfaces.FactoryObject.hash_for_dict(ins_dict):
                self._store[instrument.obj_id].append(ins_dict)
        else:
            self._store[instrument.obj_id] = [instrument.to_dict()]
            
    def get_by_id(self, obj_id: str):
        if obj_id in self._store.keys():
            return self.create(self._store[obj_id][-1])
        else:
            raise Exception('No instrument with id ' + obj_id + ' exists in storage.')
        
    def _append_values(self, results: dict, keys: list):
        for k,v in self._store.items():
            obj = v[-1]
            for r in results.keys():
                if r == 'num_version':
                    results[r].append(len(v))
                else:
                    results[r].append(obj.get(r))
        
    def get_object_list(self, keys=['obj_id', 'cls', 'expiry', 'issue_date' ]):
        num_version = []
        tmp ={k:[] for k in keys}
        tmp['num_version'] = []
        self._append_values(tmp, keys)
        return pd.DataFrame(tmp)

In [ ]:
ins_store = MemoryStorage(_instrument_create)
mkt_store = MemoryStorage(_marketdata_create)

In [ ]:
for days in range (30,90, 30):
    bond = ZeroCouponBondSpecification('BOND_'+str(days), issue_date = dt.datetime(2023,1,1), maturity_date=dt.datetime(2023,1,1) + dt.timedelta(days=days), 
                              currency=Currency.EUR, notional=10.0, issuer='Depp2', 
                            securitization_level=SecuritizationLevel.SUBORDINATED)
    ins_store.add(bond)
    
ins_store.add(PlainVanillaCouponBondSpecification('PV_BOND_'+str(days), 
                                     issue_date = dt.datetime(2023,1,1),
                                     maturity_date=dt.datetime(2025,1,2), 
                                    currency=Currency.EUR, notional=100.0, 
                                     issuer='Depp', 
                                    securitization_level=SecuritizationLevel.SUBORDINATED, 
                                    coupon = 0.05, coupon_freq='1Y', accrual_start = dt.datetime(2023,2,10)))

In [ ]:
bond_spec = PlainVanillaCouponBondSpecification('PV_BOND_'+str(days), 
                                     issue_date = dt.datetime(2023,1,1),
                                     maturity_date=dt.datetime(2023,8,2), 
                                    currency=Currency.EUR, notional=100.0, 
                                     issuer='Depp', 
                                    securitization_level=SecuritizationLevel.SUBORDINATED, 
                                    coupon = 0.05, coupon_freq='6M', accrual_start = dt.datetime(2023,1,30)) 

bond_spec.expected_cashflows()

In [ ]:
bond_pricing.DeterministicCashflowPricer.compute_yield(target_dirty_price=100.0, val_date = dt.datetime(2023,2,10), specification=bond_spec)

In [ ]:
bond_spec.expected_cashflows()

In [ ]:
tenor = Period(0,6,0)
bond_spec = FixedRateBondSpecification.from_master_data('PV_BOND_'+str(days), 
                                     issue_date = dt.datetime(2023,1,1),
                                     maturity_date=dt.datetime(2023,8,2),
                                     coupon=0.05,
                                     tenor=tenor,
                                     backwards=False,
                                     stub=True,
                                     securitisation_level=SecuritizationLevel.SUBORDINATED
                                    ) 

In [ ]:
bond_spec.coupon_payment_dates, bond_spec.coupons

In [ ]:
bond_spec.to_dict()

In [ ]:
ins_store.get_object_list(keys=['obj_id', 'cls', 'maturity_date', 'issue_date', 'issuer', 'securitisation_level' ])

In [ ]:
from rivapy.marketdata.curves import NelsonSiegel, DiscountCurveParametrized

ns = NelsonSiegel(beta0=0.001, beta1 = -0.2, beta2=-0.06, tau=1)
dc = DiscountCurveParametrized('DC',  refdate = dt.datetime(2023,1,1), rate_parametrization=ns, 
                               daycounter = DayCounterType.Act365Fixed)
mkt_store.add(dc)

In [ ]:
mkt_store.get_by_id('DC')
dates = [dt.datetime(2023,1,1) + dt.timedelta(days=30*days) for days in range(120)]
values = [dc.value(refdate = dt.datetime(2023,1,1),d=d) for d in dates]
plt.plot(dates, values)
plt.show()